# Modeling

### Importing the necessary libraries

In [3]:
pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 MB 26.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.9/295.9 MB 29.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]m1/2 [xgboost]

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install lightgbm 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 12.5 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, accuracy_score, f1_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier
import lightgbm as lgb
from sklearn.cluster import KMeans, DBSCAN

In [3]:
df1=pd.read_csv("/workspaces/Spatio-Temporal-EV-Adoption-Forecasting-Project/data/processed/EV_specs.csv")

In [4]:
file_path = "/workspaces/Spatio-Temporal-EV-Adoption-Forecasting-Project/data/processed/ev_spatial_preprocessed.csv.gz"
df2 = pd.read_csv(file_path, nrows=50000)

In [5]:
df3=pd.read_csv("/workspaces/Spatio-Temporal-EV-Adoption-Forecasting-Project/data/processed/temporal_forecast_result.csv")

In [6]:
df1.head()

,battery,efficiency,fast_charge,price.de.,range,top_speed,acceleration..0.100.,price_usd_estimated,car_name_Abarth 500e Hatchback,car_name_Aiways U5,...,car_name_link_https://ev-database.org/car/2038/Ford-Mustang-Mach-E-GT,car_name_link_https://ev-database.org/car/2039/Citroen-e-C3,car_name_link_https://ev-database.org/car/2040/BMW-iX2-xDrive30,car_name_link_https://ev-database.org/car/2041/Smart-1-Pro,car_name_link_https://ev-database.org/car/2042/Nissan-Townstar-EV-Passenger,car_name_link_https://ev-database.org/car/2044/Hyundai-IONIQ-5-N,car_name_link_https://ev-database.org/car/2045/Mercedes-EQE-SUV-300,car_name_link_https://ev-database.org/car/2046/Mercedes-EQE-SUV-350plus,car_name_link_https://ev-database.org/car/2047/Mercedes-EQE-SUV-350-4MATIC,car_name_link_https://ev-database.org/car/2048/Mercedes-EQE-SUV-500-4MATIC
0,0.188720,-0.727285,0.496939,-0.211230,0.609385,0.997205,-0.773185,-0.211238,0,0,...,0,0,0,0,0,0,0,0,0,0
1,-0.644590,-1.825666,0.624118,-0.609628,0.469416,0.554993,-0.437018,-0.609613,0,0,...,0,0,0,0,0,0,0,0,0,0
2,-0.546553,-0.382080,-0.774848,-0.659284,-0.370400,-0.578178,-0.100850,-0.659280,0,0,...,0,0,0,0,0,0,0,0,0,0
3,-0.448517,-0.758668,0.327367,-0.803582,-0.090462,-0.578178,0.235317,-0.803584,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0.188720,-1.449078,0.963261,-0.329439,1.262575,0.554993,-1.109353,-0.329424,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
df2.head()

,city,postal_code,model_year,make,model,cafv_eligibility,electric_range,base_msrp,legislative_district,dol_vehicle_id,...,state_SC,state_TN,state_TX,state_UT,state_VA,state_WA,state_WY,electric_vehicle_type_Plug-in Hybrid Electric Vehicle (PHEV),longitude,latitude
0,29447,-0.020657,-0.172448,79659,MODEL Y,0,2.523975,-0.128384,0.528607,-1.246277,...,0,0,0,0,0,1,0,0,-122,48
1,5863,-0.062009,0.831106,79659,MODEL Y,1,-0.639721,-0.128384,-1.891247,0.317125,...,0,0,0,0,0,1,0,0,-122,48
2,29447,-0.025979,-0.506966,79659,MODEL S,0,2.295667,-0.128384,0.461389,-0.836630,...,0,0,0,0,0,1,0,0,-122,48
3,2764,-0.059553,-1.510520,79659,MODEL S,0,1.643359,-0.128384,-1.622374,-0.726810,...,0,0,0,0,0,1,0,0,-122,48
4,69,0.089890,0.162070,79659,MODEL Y,1,-0.639721,-0.128384,-0.412447,-0.198982,...,0,0,0,0,0,1,0,0,-123,48


In [8]:
df3.head()

,user_id,vehicle_model,battery_capacity_(kwh),charging_station_id,charging_station_location,charging_start_time,charging_end_time,energy_consumed_(kwh),charging_duration_(hours),charging_rate_(kw),...,charging_duration_lag_2,energy_consumed_roll_mean_3,energy_consumed_roll_std_3,charging_duration_roll_mean_3,charging_duration_roll_std_3,start_hour_sin,start_hour_cos,weekday_sin,weekday_cos,time_since_last_charge
0,-1.730739,0,1.623166,1.030207,1,1969-12-31 23:59:59.999999999,-1.181028,0.839652,-1.192647,0.735299,...,-0.265530,0.047603,0.003811,-0.420515,0.091498,-0.421350,0.906898,-0.972958,0.230983,0
1,-1.728115,2,1.235132,1.287279,4,1969-12-31 23:59:59.999999999,-1.180489,-1.404045,0.658375,0.369203,...,-0.265530,0.047603,0.003811,-0.420515,0.091498,-0.386758,0.922181,-0.972958,0.230983,0
2,-1.725491,1,0.022525,-0.428847,4,1969-12-31 23:59:59.999999999,-1.180186,-1.083517,-0.267136,0.149545,...,-1.191373,-0.860889,0.631834,-0.420515,0.091498,-0.351613,0.936146,-0.972958,0.230983,0
3,-1.722866,2,-1.190081,0.585543,1,1969-12-31 23:59:59.999999999,-1.179787,1.663867,-1.192647,0.515641,...,0.660313,-0.406643,1.783210,-0.420515,0.091498,-0.315964,0.948771,-0.972958,0.230983,0
4,-1.720242,2,-1.190081,-0.936043,2,1969-12-31 23:59:59.999999999,-1.179984,-1.037727,-0.267136,-1.168403,...,-0.265530,-0.255228,1.573869,-0.420515,0.091498,-0.279864,0.960040,-0.972958,0.230983,0


In [9]:
df1.isnull().sum().sum()

np.int64(0)

In [10]:
df2.isnull().sum().sum()

np.int64(0)

In [11]:
df3.isnull().sum().sum()

np.int64(0)

In [12]:
def evaluate_regression(y_true, y_pred):
    print("R2:", r2_score(y_true, y_pred))
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print("RMSE:", rmse)

# Classification evaluation
def evaluate_classification(y_true, y_pred, average='macro'):
    """
    y_true: true labels
    y_pred: predicted labels
    average: 'binary', 'micro', 'macro', 'weighted', or None
    """
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("F1-score:", f1_score(y_true, y_pred, average=average))


1️⃣ df1 – EV Specs / Vehicle Prices 

Goal : Predict the estimated price (price_usd_estimated) of vehicles using their specs.provide the code for this

In [14]:
X = df1.drop(columns=["price_usd_estimated"])
y = df1["price_usd_estimated"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)
preds = model.predict(X_test)
print("Price Prediction (Regression)")
evaluate_regression(y_test, preds)

Price Prediction (Regression)
R2: 0.995798298039678
RMSE: 0.06865898828069679


df2 – EV Adoption (Spatial)

In [16]:
X = df2.drop(columns=["cafv_eligibility"])
y = df2["cafv_eligibility"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [17]:
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)
preds = clf.predict(X_test)
print("Adoption Eligibility (Classification)")
evaluate_classification(y_test, preds)

ValueError: could not convert string to float: 'LEAF'

In [ ]:
# 4. df3 – Charging Behavior
# =========================
print("\n--- DF3: Charging Behavior ---")

X = df3.drop(columns=["energy_consumed_(kwh)"])
y = df3["energy_consumed_(kwh)"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
reg = XGBRegressor(random_state=42)
reg.fit(X_train, y_train)
preds = reg.predict(X_test)
print("Energy Consumed Prediction (Regression)")
evaluate_regression(y_test, preds)

# Unsupervised clustering
kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(X)
print("KMeans Clusters:", np.unique(clusters))



--- DF1: Vehicle Specs ---
Price Prediction (Regression)
R2: 0.9960788172601577
RMSE: 0.06632745278258466

--- DF2: EV Adoption ---
Adoption Eligibility (Classification)
Accuracy: 1.0


ValueError: Target is multiclass but average='binary'. Please choose another average setting, one of [None, 'micro', 'macro', 'weighted'].

In [ ]:
# 5. df4 – Failure Analysis
# =========================
print("\n--- DF4: Failure Analysis ---")

X = df4.drop(columns=["failure_label"])
y = df4["failure_label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = lgb.LGBMClassifier(random_state=42)
clf.fit(X_train, y_train)
preds = clf.predict(X_test)
print("Failure Prediction (Classification)")
evaluate_classification(y_test, preds)

# =========================
# 6. Wrap up
# =========================
print("\nModeling completed for all datasets ✅")
